[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_09/listing_9.6.ipynb)

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U vllm huggingface_hub


### Listing 9.6: Serving multiple LoRA adapters dynamically on a single base model

In [ ]:
from huggingface_hub import snapshot_download
from vllm import LLM, SamplingParams 
from vllm.lora.request import LoRARequest

sql_adapter_path = snapshot_download(repo_id="predibase/wikisql") 
support_adapter_path = snapshot_download(repo_id="predibase/customer_support")

llm = LLM(
    model="mistralai/Mistral-7B-v0.1", 
    enable_lora=True,
    max_loras=2,  # Specify this if you plan to keep multiple LoRAs in memory
    gpu_memory_utilization=0.8
) 

sampling_params = SamplingParams(temperature=0.0, max_tokens=128) 

sql_prompt = "[INST] Write a SQL query to find all users whose account status is active and joined after 2023. [/INST]" 
support_prompt = "[INST] You are a helpful customer support agent. Respond to this: 'My package arrived completely crushed, I want a refund!' [/INST]"

print("--- Using SQL Adapter ---")
outputs_sql = llm.generate( 
    [sql_prompt], 
    sampling_params, 
    lora_request=LoRARequest(lora_name="sql_adapter", lora_int_id=1, lora_path=sql_adapter_path) 
) 
print(outputs_sql[0].outputs[0].text.strip()) 

print("\n--- Using Customer Support Adapter ---")
outputs_support = llm.generate( 
    [support_prompt], 
    sampling_params, 
    lora_request=LoRARequest(lora_name="support_adapter", lora_int_id=2, lora_path=support_adapter_path) 
) 
print(outputs_support[0].outputs[0].text.strip())

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

INFO 05-04 13:21:56 [utils.py:233] non-default args: {'disable_log_stats': True, 'enable_lora': True, 'max_loras': 2, 'model': 'mistralai/Mistral-7B-v0.1'}
INFO 05-04 13:21:58 [model.py:555] Resolved architecture: MistralForCausalLM
INFO 05-04 13:21:58 [model.py:1680] Using max model len 32768
INFO 05-04 13:21:58 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-04 13:21:58 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-04 13:21:58 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=49112) INFO 05-04 13:22:00 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='mistralai/Mistral-7B-v0.1', speculative_config=None, tokenizer='mistralai/Mistral-7B-v0.1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=49112) INFO 05-04 13:24:21 [default_loader.py:384] Loading weights took 133.92 seconds
(EngineCore pid=49112) INFO 05-04 13:24:21 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore pid=49112) INFO 05-04 13:24:21 [gpu_model_runner.py:4879] Model loading took 13.65 GiB memory and 139.540211 seconds
(EngineCore pid=49112) INFO 05-04 13:24:27 [backends.py:1069] Using cache directory: /home/lmassaron/.cache/vllm/torch_compile_cache/ed8cd04e6e/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=49112) INFO 05-04 13:24:27 [backends.py:1128] Dynamo bytecode transform time: 5.74 s


...

SELECT COUNT ID FROM table WHERE Status = active AND JoinDate > 2023

--- Using Customer Support Adapter ---


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

My package arrived completely crushed, I want a refund!

I'm sorry to hear that your package arrived crushed. We take great care in packaging our products to ensure they arrive in perfect condition, but unfortunately accidents can happen.

I understand that you want a refund, and we want to make sure you're happy with your purchase. However, we do have a return policy in place that requires the product to be returned in its original condition.

If you're willing to return the product in its original condition, we can issue a refund. If not, we'll be happy to work with


(EngineCore pid=49112) INFO 05-04 13:26:40 [core.py:1238] Shutdown initiated (timeout=0)
(EngineCore pid=49112) INFO 05-04 13:26:40 [core.py:1261] Shutdown complete


In [2]:
%%bash
pkill VLLM